In [ ]:
import numpy as np
import pandas as pd
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torch.nn.functional as F
import torchvision.datasets as datasets
from torchvision import models
from torch.utils.data import DataLoader, Dataset, random_split, Subset
from torch.utils.data.distributed import DistributedSampler
from tqdm import tqdm
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
from torch_xla import runtime as xr
import scipy.io
import xml.etree.ElementTree as ET
from PIL import Image
#import torchmetrics
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="jax")
os.environ.pop('TPU_PROCESS_ADDRESSES')

In [ ]:
train_list = scipy.io.loadmat('/kaggle/input/datasets/devvrath123/stanford-dogs-split-data/train_list.mat')
test_list = scipy.io.loadmat('/kaggle/input/datasets/devvrath123/stanford-dogs-split-data/test_list.mat')
path = '/kaggle/input/datasets/jessicali9530/stanford-dogs-dataset/images/Images/'

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

"""
validation_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])
"""

validation_transforms=transforms.Compose([
    transforms.Resize(236, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_full = datasets.ImageFolder(root=path, transform=train_transforms)
val_full = datasets.ImageFolder(root=path, transform=validation_transforms)

train_filenames = [f[0][0] for f in train_list['file_list']]
test_filenames = [f[0][0] for f in test_list['file_list']]

all_samples = train_full.samples
path_to_idx = {os.path.join(os.path.basename(os.path.dirname(path)), os.path.basename(path)): i
               for i, (path, _) in enumerate(all_samples)}

train_indices = [path_to_idx[f] for f in train_filenames if f in path_to_idx]
val_indices = [path_to_idx[f] for f in test_filenames if f in path_to_idx]

training = Subset(train_full, train_indices)
validation = Subset(val_full, val_indices)

print("Official Split:")
print(f"Training: {len(training)}, Validation: {len(validation)}")

Official Split:
Training: 12000, Validation: 8580


In [ ]:
def train_model(model, train_loader, train_sampler, val_loader, criterion, optimizer, scheduler, num_epochs):
    is_master = xm.is_master_ordinal()
    for epoch in range(num_epochs):
        model.train()
        train_sampler.set_epoch(epoch)
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", unit="batch", disable=not is_master)
        for inputs, labels in train_pbar:
            #inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            xm.optimizer_step(optimizer)

            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += torch.sum(preds == labels.data).item()
            train_total += inputs.size(0)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", unit="batch", leave=True, disable=not is_master)
            for inputs, labels in val_pbar:
                #inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels.data).item()
                val_total += inputs.size(0)

        """
        train_correct = xm.mesh_reduce('train_correct', torch.tensor(train_correct), sum)
        train_total = xm.mesh_reduce('train_total', torch.tensor(train_total), sum)
        val_correct = xm.mesh_reduce('val_correct', torch.tensor(val_correct), sum)
        val_total = xm.mesh_reduce('val_total', torch.tensor(val_total), sum)

        train_loss = xm.mesh_reduce('train_loss', torch.tensor(train_loss), sum)
        val_loss = xm.mesh_reduce('val_loss', torch.tensor(val_loss), sum)
        avg_train_loss = train_loss.item() / train_total
        avg_val_loss = val_loss.item() / val_total

        t_acc = 100 * train_correct.double() / train_total
        v_acc = 100 * val_correct.double() / val_total
        """

        metrics = torch.tensor(
            [train_correct, train_total, val_correct, val_total, train_loss, val_loss],
            dtype=torch.float64,
            device=torch_xla.device()
        )

        synced_metrics = xm.all_reduce(xm.REDUCE_SUM, metrics)
        tc, tt, vc, vt, tl, vl = synced_metrics.tolist()

        avg_train_loss = tl / tt
        avg_val_loss = vl / vt
        t_acc = 100 * tc / tt
        v_acc = 100 * vc / vt

        scheduler.step(v_acc)

        if is_master:
            print(f"Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")
            print(f"Training Accuracy: {t_acc:.2f}%, Validation Accuracy: {v_acc:.2f}%")

In [ ]:
def fine_tune(model, train_loader, train_sampler, val_loader, criterion, num_epochs, lr):
    for parameter in model.parameters():
        parameter.requires_grad = True
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=4)
    if xm.is_master_ordinal():
        print('\nFine Tuning:\n')
    train_model(model, train_loader, train_sampler, val_loader, criterion, optimizer, scheduler, num_epochs)

In [ ]:
def _mp_fn(index):
    device = torch_xla.device()
    lr1 = 0.002
    lr2 = 0.00002

    train_sampler = DistributedSampler(training, num_replicas=xr.world_size(), rank=xr.global_ordinal(), shuffle=True)
    validation_sampler = DistributedSampler(validation, num_replicas=xr.world_size(), rank=xr.global_ordinal(), shuffle=False)

    train_loader = DataLoader(training, batch_size=32, sampler=train_sampler, num_workers=8, shuffle=False)
    validation_loader = DataLoader(validation, batch_size=32, sampler=validation_sampler, num_workers=8, shuffle=False)

    train_loader_xla = pl.MpDeviceLoader(train_loader, device)
    validation_loader_xla = pl.MpDeviceLoader(validation_loader, device)
    #model = models.convnext_base(weights='DEFAULT')
    model = models.convnext_small(weights='DEFAULT')
    for parameter in model.parameters():
        parameter.requires_grad = False

    model.classifier[2] = nn.Linear(model.classifier[2].in_features, 120)
    model.to(device)
    xm.broadcast_master_param(model)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    optimizer = optim.Adam(model.classifier[2].parameters(), lr=lr1)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=4)
    #scheduler2 = optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

    train_model(model, train_loader_xla, train_sampler, validation_loader_xla, criterion, optimizer, scheduler, num_epochs=25)
    fine_tune(model, train_loader_xla, train_sampler, validation_loader_xla, criterion, num_epochs=25, lr=lr2)

    if xm.is_master_ordinal():
        print("Saving model")

    xm.save(model.state_dict(), 'dog-classifier-ConvNextSmall3-of_spt_no_annot.pth')

In [ ]:
torch_xla.launch(_mp_fn, args=(), start_method='fork')

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:00<00:00, 440MB/s]
Epoch 1 [Val]: 100%|██████████| 269/269 [02:57<00:00,  1.52batch/s]


Training Loss: 1.6733, Validation Loss: 1.0323
Training Accuracy: 77.72%, Validation Accuracy: 95.17%


Epoch 2 [Val]: 100%|██████████| 269/269 [00:15<00:00, 16.93batch/s]


Training Loss: 1.3682, Validation Loss: 1.0388
Training Accuracy: 83.28%, Validation Accuracy: 95.54%


...

Training Loss: 1.1037, Validation Loss: 0.9650
Training Accuracy: 92.03%, Validation Accuracy: 95.78%


Epoch 24 [Val]: 100%|██████████| 269/269 [00:18<00:00, 14.21batch/s]


Training Loss: 1.1003, Validation Loss: 0.9649
Training Accuracy: 92.05%, Validation Accuracy: 95.78%


Epoch 25 [Val]: 100%|██████████| 269/269 [00:18<00:00, 14.58batch/s]


Training Loss: 1.1129, Validation Loss: 0.9646
Training Accuracy: 91.62%, Validation Accuracy: 95.78%
Saving model
